[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sterolose/sirius-statistics-2026/blob/main/notebooks/3_categorical.ipynb)

## Семинар 3. Критерии согласия и категориальные данные

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

### Задание 1

При скрещивании двух гетерозигот `Aa` ожидаем генотипы `AA`, `Aa` и `aa` в соотношении 1:2:1. Это предполагает обычное менделевское наследование и одинаковую вероятность выживания и обнаружения всех генотипов. Генотипировали 200 независимо отобранных потомков.

In [ ]:
df = pd.DataFrame({
    "probability": [0.25, 0.50, 0.25],
    "expected": [50, 100, 50],
    "observed": [46, 101, 53],
}, index=["AA", "Aa", "aa"])
df

Для оценки правдоподнобности получения такого распределения по генотипам может быть применен $χ^2$-критерий согласия Пирсона.

In [ ]:
result = stats.chisquare(df["observed"], df["expected"])
pd.Series({
    "statistic": result.statistic,
    "p-value": result.pvalue,
})

В другом скрещивании получили 64 особи `AA`, 98 — `Aa` и 38 — `aa`. Проверьте согласие со вторым законом Менделя.

In [ ]:
# Your code here

### Задание 2

В пилотном исследовании 40 пациентов с бактериальной инфекцией случайно разделили на две группы по 20 человек. Все получали стандартную терапию; в группе `Drug` дополнительно назначили экспериментальный препарат, в группе `Control` — плацебо. Через 7 дней оценили улучшение состояния по заранее заданному клиническому критерию. Каждая строка относится к отдельному пациенту, наблюдения в выборке независимы.

In [ ]:
df = pd.DataFrame({
    "treatment": np.repeat(["Drug", "Drug", "Control", "Control"], [14, 6, 7, 13]),
    "outcome": np.repeat(["Improved", "Not improved", "Improved", "Not improved"], [14, 6, 7, 13]),
})
table = pd.crosstab(df["treatment"], df["outcome"])
table

Независимость исхода от лечения можно оценить с помощью `stats.chi2_contingency`.

In [ ]:
result = stats.chi2_contingency(table)
pd.Series({
    "statistic": result.statistic,
    "p-value": result.pvalue,
})

Было проведено исследование альтернативного препарата. Таблица сопряженности уже составлена. Повторите тест для нее. Отвергаем ли мы нулевую гипотезу при уровне значимости 0.05? А при 0.01?

In [ ]:
table = pd.DataFrame(
    [[7, 13], [16, 4]],
    index=["Control", "Drug"],
    columns=["Improved", "Not improved"],
)
table

In [ ]:
# Your code here

### Задание 3

Для той же таблицы применим точный тест Фишера. Тест не использует $χ^2$-приближение и подходит в том числе для малых ожидаемых численностей.

In [ ]:
result = stats.fisher_exact(table)
pd.Series({
    "statistic": result.statistic,
    "p-value": result.pvalue,
})

На малой выборке был исследован комбинированный препарат из первых двух лекарств. Используя приведенную таблицу сопряженности, выполните `chi2_contingency` и `fisher_exact`. Сравните результат. Почему вывод может расходиться?

In [ ]:
table = pd.DataFrame(
    [[6, 3], [1, 8]],
    index=["Drug", "Control"],
    columns=["Improved", "Not improved"],
)
table

In [ ]:
# Your code here

### Задание 4

Разработаем свой пермутационный тест. В качестве статистики возьмем разность долей улучшений:
$$\Delta = \hat p_{\mathrm{Drug}} - \hat p_{\mathrm{Control}}.$$

In [ ]:
df = pd.DataFrame({
    "treatment": np.repeat(["Drug", "Drug", "Control", "Control"], [19, 11, 12, 18]),
    "outcome": np.repeat(["Improved", "Not improved", "Improved", "Not improved"],[19, 11, 12, 18]),
})
table = pd.crosstab(df["treatment"], df["outcome"])
table

In [ ]:
treatment = df["treatment"]
improved = df["outcome"] == "Improved"

drug_rate = improved[treatment == "Drug"].mean()
control_rate = improved[treatment == "Control"].mean()
observed_difference = drug_rate - control_rate

pd.Series({
    "Drug improvement rate": drug_rate,
    "Control improvement rate": control_rate,
    "Difference": observed_difference,
})

При нулевой гипотезе метки лечения обменяемы: их можно переставлять между пациентами, сохраняя исходы и размеры групп. В этом примере такую перестановку обосновывают случайное распределение по группам и независимость наблюдений. Осуществим одну случайную перестановку.

In [ ]:
rng = np.random.default_rng(42)
shuffled_treatment = rng.permutation(treatment)

shuffled_difference = (
    improved[shuffled_treatment == "Drug"].mean()
    - improved[shuffled_treatment == "Control"].mean()
)

(observed_difference, shuffled_difference)

Осуществите 10000 перестановок, рассчитав для каждой значение статистики. Запишите их в список `permuted_differences`.

In [ ]:
rng = np.random.default_rng(42)
n_permutations = 10000

# Your code here

Постройте гистограмму `permuted_differences`. Добавьте вертикальные линии в точках `observed_difference` и `-observed_difference`. Где находится центр распределения? Насколько далеко от него наблюдаемый эффект?

In [ ]:
# Your code here

Для двустороннего теста посчитайте число перестановок $b$, для которых $|\Delta_{\mathrm{perm}}| \geq |\Delta_{\mathrm{obs}}|$. Оцените p-value по формуле:

$$\hat p = \frac{b+1}{10000+1}.$$

Сравните результат с тестом Фишера и $χ^2$-критерием.

In [ ]:
# Your code here